# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adnan-ai98/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Ranking**

My lane is **Refresh / Content Opportunity Scoring**, and I will frame it as a **ranking problem**.

The goal is to rank content pages by how strongly the available evidence suggests that they should be reviewed first for actions such as refresh, expansion, protection, pruning, or monitoring.

This is a ranking problem because the practical decision is not simply whether a page is good or bad. A content team has limited review capacity, so the useful output is an ordered list of pages, with the highest-priority candidates at the top.



In [26]:
lane = "Refresh / Content Opportunity Scoring"
task_type = "Ranking"

print("Lane:", lane)
print("Task type:", task_type)

Lane: Refresh / Content Opportunity Scoring
Task type: Ranking


## 2. Target or proxy

**Target / proxy: `is_declining_label`**

For the starter version of this task, I will use `is_declining_label` as a proxy for identifying pages that may need attention.

The proxy is defined from the observed `trend_direction` field:

`is_declining_label = 1` when `trend_direction == "down"`, otherwise `0`.

This is a **proxy**, not a true future outcome. It describes the current observed direction in the available data rather than predicting whether a page will decline in a future period.

A stronger future version could use information from an earlier time window to predict an observed decline or recovery in a later time window.



In [27]:
target_proxy = "is_declining_label"
proxy_definition = 'trend_direction == "down"'

print("Target/proxy:", target_proxy)
print("Definition:", proxy_definition)

Target/proxy: is_declining_label
Definition: trend_direction == "down"


## 3. Success metric

**Success metric: Precision@50**

I will use **Precision@50** as the main success metric.

Precision@50 measures how many of the top 50 ranked pages match the positive target/proxy.

This metric fits the real decision because a content team has limited review capacity. If the model produces a ranked review queue, we want as many of the top 50 pages as possible to be relevant candidates for review.

A higher Precision@50 means the review team can spend more of its limited capacity on pages that match the target/proxy.


In [28]:
success_metric = "Precision@50"
review_capacity = 50

print("Success metric:", success_metric)
print("Review capacity:", review_capacity, "pages")

Success metric: Precision@50
Review capacity: 50 pages


## 4. The unit of analysis, as a real dataframe

**Unit of analysis: one row = one content page**

For this lane, the unit of analysis is a **content page**.

**One row = one content page.**

Each row represents one content item and contains measurable signals about its search visibility, traffic, engagement, age, freshness, and trend.

For this starter analysis, I will keep pages with `impressions_90d > 0` and `content_age_days >= 90`. I will also remove duplicate `content_id` values so that each row represents one content page.

The target/proxy column is `is_declining_label`, where `1` means the observed `trend_direction` is `"down"` and `0` means it is not `"down"`.

The eventual ML system would use the available page-level signals to produce a ranked review priority.


In [29]:
from pathlib import Path
import pandas as pd

data_path = Path(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

# Keep pages with measurable impressions and sufficient content age
lane_df = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].copy()

# One row = one content page
lane_df = lane_df.drop_duplicates(subset="content_id").copy()

# Create the starter proxy target
lane_df["is_declining_label"] = (
    lane_df["trend_direction"] == "down"
).astype(int)

# Show the actual page-level dataframe
display(
    lane_df[
        [
            "content_id",
            "client_id",
            "impressions_90d",
            "sessions_90d",
            "content_age_days",
            "trend_direction",
            "is_declining_label"
        ]
    ].head(10)
)

print("Lane dataframe shape:", lane_df.shape)

print("\nTarget distribution:")
print(lane_df["is_declining_label"].value_counts())

Dataset shape: (30000, 44)


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,down,1
6,content_9a34b442b552,client_8722616204,20,1,90,down,1
7,content_a63219c6e95a,client_19581e27de,1724,28,445,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,68,90,down,1
9,content_c27558df2b0c,client_19581e27de,1240,3,257,down,1


Lane dataframe shape: (30000, 45)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 5. Why ML beats a fixed rule here

**Why ML beats a fixed rule**

A fixed rule such as `if impressions_90d < 500: review` would only use one manually selected threshold. This could miss pages that have high visibility but are declining, have low engagement, are outdated, or have other signals that indicate a review opportunity.

The opportunity to review a page depends on multiple signals, including impressions, clicks, sessions, CTR, average position, engagement rate, content age, freshness, and recent trends.

ML can learn patterns from combinations of these signals instead of relying on one manually chosen threshold. It can then produce a ranked list of pages that are more likely to be useful candidates for review.

The model would support a human decision rather than automatically deciding what should happen to a page. A content team could use the ranked output to prioritize pages for refresh, expansion, protection, pruning, or monitoring.

The ML approach should be compared with a simple fixed-rule baseline to determine whether learning from multiple signals provides useful improvement.



In [30]:
signals = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]

print("Signals that could inform refresh opportunity:")
for signal in signals:
    print("-", signal)

print("\nML goal: combine multiple signals to prioritize content pages for human review.")

Signals that could inform refresh opportunity:
- impressions_90d
- clicks_90d
- sessions_90d
- ctr
- avg_position
- engagement_rate
- content_age_days
- days_since_last_update
- trend_pct

ML goal: combine multiple signals to prioritize content pages for human review.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.